[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch16_multi_agent_ru.ipynb)

<div style="background: linear-gradient(135deg, #0d2137 0%, #1a3a5c 50%, #1b5e20 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #69f0ae; font-size: 2.2em; margin: 0;">Глава 16. Многоагентное обучение с подкреплением</h1>
  <p style="color: #b0bec5; font-size: 1.1em; margin-top: 10px;">Итерированная дилемма заключённого · Tit-for-Tat · self-play · нетранзитивность · кооперативные задачи</p>
</div>

Ноутбук реализует четыре эксперимента многоагентного RL:
1. **Итерированная дилемма заключённого** с Q-обучением.
2. **Возникновение Tit-for-Tat** через self-play.
3. **Self-play в «камень-ножницы-бумага»** и нетранзитивность.
4. **Кооперативная задача** с общим вознаграждением.

## Подготовка окружения

> **Colab/Kaggle**: запускается на бесплатном CPU. GPU не требуется.
> Ожидаемое время: ~3 минуты от начала до конца.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
from typing import List, Tuple, Dict
import random

np.random.seed(42)
random.seed(42)
print("Imports OK")

<div style="background: #1b2838; padding: 16px; border-radius: 8px; border-left: 4px solid #69f0ae;">
  <h2 style="color: #69f0ae; margin: 0;">Часть 1 — Итерированная дилемма заключённого с Q-обучением</h2>
</div>

**Матрица выигрышей дилеммы заключённого** (каждый агент выбирает C — кооперировать, D — отступать):

|            | Партнёр C | Партнёр D |
|------------|-----------|-----------|
| **Агент C** | (3, 3)    | (0, 5)    |
| **Агент D** | (5, 0)    | (1, 1)    |

Равновесие Нэша одного раунда — (D, D), но в *итерированной* версии может возникнуть кооперация.

In [ ]:
# Prisoner's Dilemma payoffs
# Actions: 0 = Cooperate, 1 = Defect
PD_PAYOFF = np.array([
    [(3, 3), (0, 5)],
    [(5, 0), (1, 1)]
])  # PD_PAYOFF[a1][a2] = (reward1, reward2)


class QLearningAgent:
    """Tabular Q-learning agent for iterated games.
    State = (my_last_action, opponent_last_action), or 'start'.
    """

    def __init__(self, n_actions: int = 2, alpha: float = 0.1,
                 gamma: float = 0.9, epsilon: float = 0.1, name: str = ""):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.name = name
        # Q-table: state -> action -> value
        # States: (my_last, opp_last) pairs + initial state (-1, -1)
        self.Q = defaultdict(lambda: np.zeros(n_actions))

    def act(self, state) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return int(np.argmax(self.Q[state]))

    def update(self, state, action, reward, next_state):
        td_error = reward + self.gamma * np.max(self.Q[next_state]) - self.Q[state][action]
        self.Q[state][action] += self.alpha * td_error

    def cooperate_rate(self) -> float:
        """Average probability of cooperation across all states."""
        rates = []
        for state, q_vals in self.Q.items():
            if q_vals.sum() == 0:
                continue
            # Soft max-argmax
            rates.append(float(np.argmax(q_vals) == 0))
        return np.mean(rates) if rates else 0.5


def run_ipd(n_episodes: int = 10000, n_rounds_per_ep: int = 20,
             epsilon: float = 0.1) -> Tuple[List, List, List, List]:
    """Run iterated Prisoner's Dilemma and record cooperation rates."""
    agent1 = QLearningAgent(epsilon=epsilon, name="Agent1")
    agent2 = QLearningAgent(epsilon=epsilon, name="Agent2")

    coop1_hist, coop2_hist = [], []
    reward1_hist, reward2_hist = [], []

    for ep in range(n_episodes):
        state1 = state2 = (-1, -1)  # initial state
        ep_r1, ep_r2 = 0.0, 0.0
        ep_a1, ep_a2 = [], []

        for _ in range(n_rounds_per_ep):
            a1 = agent1.act(state1)
            a2 = agent2.act(state2)
            r1, r2 = PD_PAYOFF[a1][a2]
            next_state1 = (a1, a2)
            next_state2 = (a2, a1)  # agent2 sees its own action first
            agent1.update(state1, a1, r1, next_state1)
            agent2.update(state2, a2, r2, next_state2)
            state1, state2 = next_state1, next_state2
            ep_r1 += r1; ep_r2 += r2
            ep_a1.append(a1); ep_a2.append(a2)

        coop1_hist.append(ep_a1.count(0) / n_rounds_per_ep)
        coop2_hist.append(ep_a2.count(0) / n_rounds_per_ep)
        reward1_hist.append(ep_r1 / n_rounds_per_ep)
        reward2_hist.append(ep_r2 / n_rounds_per_ep)

    return coop1_hist, coop2_hist, reward1_hist, reward2_hist


print("Running iterated Prisoner's Dilemma (10k episodes)...")
c1, c2, r1, r2 = run_ipd(n_episodes=10000)
print(f"Final 1000 ep cooperation rate: Agent1={np.mean(c1[-1000:]):.1%}, Agent2={np.mean(c2[-1000:]):.1%}")
print(f"Final 1000 ep mean reward: Agent1={np.mean(r1[-1000:]):.2f}, Agent2={np.mean(r2[-1000:]):.2f}")

In [ ]:
# Plot cooperation rates over training
def smooth(x, w=200): return np.convolve(x, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')

# Cooperation rate
ax = axes[0]
ax.set_facecolor('#0f1117')
x = np.arange(len(smooth(c1)))
ax.plot(x, smooth(c1), color='#69f0ae', lw=2, label='Agent 1')
ax.plot(x, smooth(c2), color='#40c4ff', lw=2, linestyle='--', label='Agent 2')
ax.axhline(0.5, color='gray', linestyle=':', lw=1)
ax.set_xlabel('Episode'); ax.set_ylabel('Cooperation Rate')
ax.set_title('Cooperation Rate over Training', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

# Reward
ax = axes[1]
ax.set_facecolor('#0f1117')
ax.plot(x, smooth(r1), color='#ffb74d', lw=2, label='Agent 1')
ax.plot(x, smooth(r2), color='#ef9a9a', lw=2, linestyle='--', label='Agent 2')
ax.axhline(1.0, color='#e94560', linestyle=':', lw=1, label='(D,D) payoff=1')
ax.axhline(3.0, color='#69f0ae', linestyle=':', lw=1, label='(C,C) payoff=3')
ax.set_xlabel('Episode'); ax.set_ylabel('Mean Reward per Round')
ax.set_title('Mean Reward over Training', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=8)
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

plt.suptitle("Iterated Prisoner's Dilemma: Q-Learning Agents", color='white', fontsize=14)
plt.tight_layout()
plt.show()

<div style="background: #1b2838; padding: 16px; border-radius: 8px; border-left: 4px solid #ffb74d;">
  <h2 style="color: #ffb74d; margin: 0;">Часть 2 — Возникновение Tit-for-Tat</h2>
</div>

**Tit-for-Tat (TfT)**: всегда кооперировать в первом ходе, затем копировать последнее действие оппонента.
Проверим, что Q-агент против TfT учится кооперировать, и что Q-обучение vs Q-обучение тоже может сойтись к взаимной кооперации.

In [ ]:
class TitForTatAgent:
    """Classic Tit-for-Tat: cooperate first, then mirror opponent."""

    def __init__(self):
        self.last_opponent_action = 0  # start cooperating

    def act(self, state) -> int:
        return self.last_opponent_action

    def observe_opponent(self, opp_action: int):
        self.last_opponent_action = opp_action

    def update(self, *args): pass  # no learning


def run_qlearner_vs_tft(n_episodes: int = 5000, n_rounds: int = 20) -> List[float]:
    """Q-learning agent vs Tit-for-Tat."""
    qagent = QLearningAgent(epsilon=0.1)
    tft    = TitForTatAgent()
    coop_hist = []

    for ep in range(n_episodes):
        state = (-1, -1)
        ep_a = []
        for _ in range(n_rounds):
            a_q   = qagent.act(state)
            a_tft = tft.act(state)
            r_q, _ = PD_PAYOFF[a_q][a_tft]
            next_state = (a_q, a_tft)
            qagent.update(state, a_q, r_q, next_state)
            tft.observe_opponent(a_q)
            state = next_state
            ep_a.append(a_q)
        coop_hist.append(ep_a.count(0) / n_rounds)
    return coop_hist


coop_vs_tft = run_qlearner_vs_tft(5000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
sm = smooth(coop_vs_tft, 100)
ax.plot(sm, color='#69f0ae', lw=2)
ax.axhline(1.0, color='white', linestyle=':', lw=1, label='Full cooperation')
ax.set_xlabel('Episode'); ax.set_ylabel('Q-Agent Cooperation Rate')
ax.set_title('Q-Learning vs Tit-for-Tat: Agent Learns to Cooperate', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout()
plt.show()
print(f"Final cooperation rate (last 500 ep): {np.mean(coop_vs_tft[-500:]):.1%}")

<div style="background: #1b2838; padding: 16px; border-radius: 8px; border-left: 4px solid #ce93d8;">
  <h2 style="color: #ce93d8; margin: 0;">Часть 3 — Self-play в «камень-ножницы-бумага»</h2>
</div>

«Камень-ножницы-бумага» имеет единственное равновесие Нэша: равномерная случайная игра.
В **self-play** агент играет против копии себя. Покажем:
- Обученная стратегия сходится примерно к равномерной.
- **Нетранзитивность**: стратегия, побеждающая стратегию A, может проиграть стратегии B, которая проигрывает стратегии A.

In [ ]:
# Rock=0, Paper=1, Scissors=2
RPS_REWARD = np.array([
    [ 0, -1,  1],
    [ 1,  0, -1],
    [-1,  1,  0]
])  # reward for row player


class RPSAgent:
    """Q-learning agent for RPS. State = opponent's last action (or -1)."""

    def __init__(self, alpha=0.05, gamma=0.0, epsilon=0.1):
        self.alpha = alpha; self.gamma = gamma; self.epsilon = epsilon
        self.Q = defaultdict(lambda: np.zeros(3))

    def act(self, state) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, 2)
        return int(np.argmax(self.Q[state]))

    def update(self, state, action, reward, next_state):
        td = reward + self.gamma * np.max(self.Q[next_state]) - self.Q[state][action]
        self.Q[state][action] += self.alpha * td

    def policy(self, state=-1) -> np.ndarray:
        q = self.Q[state]
        # Softmax
        e = np.exp(q - q.max())
        return e / e.sum()


def run_rps_selfplay(n_episodes: int = 20000) -> Tuple[List, RPSAgent]:
    agent = RPSAgent(alpha=0.05, epsilon=0.05)
    policy_hist = []

    for ep in range(n_episodes):
        state = -1
        # Self-play: agent plays against a frozen snapshot (re-use same agent for simplicity)
        a1 = agent.act(state)
        a2 = agent.act(state)   # opponent = same agent independently acting
        r1 = RPS_REWARD[a1][a2]
        r2 = -r1
        agent.update(state, a1, r1, a2)
        if ep % 200 == 0:
            policy_hist.append(agent.policy(-1).copy())

    return policy_hist, agent


print("Running RPS self-play (20k episodes)...")
policy_hist, rps_agent = run_rps_selfplay(20000)

# Plot policy evolution
labels = ['Rock', 'Paper', 'Scissors']
colors_rps = ['#ef5350', '#66bb6a', '#42a5f5']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')

ax = axes[0]
ax.set_facecolor('#0f1117')
ph = np.array(policy_hist)
for i, (label, color) in enumerate(zip(labels, colors_rps)):
    ax.plot(ph[:, i], color=color, lw=2, label=label)
ax.axhline(1/3, color='white', linestyle=':', lw=1, label='Nash (1/3)')
ax.set_xlabel('Episode (×200)'); ax.set_ylabel('Action Probability')
ax.set_title('RPS Self-Play: Policy Convergence', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

# Final policy bar chart
ax = axes[1]
ax.set_facecolor('#0f1117')
final_p = rps_agent.policy(-1)
bars = ax.bar(labels, final_p, color=colors_rps, edgecolor='white')
ax.axhline(1/3, color='white', linestyle='--', lw=1.5, label='Nash (1/3)')
for bar, v in zip(bars, final_p):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.2f}', ha='center', color='white', fontweight='bold')
ax.set_ylabel('Probability'); ax.set_ylim(0, 0.6)
ax.set_title('Final Learned Policy', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

plt.suptitle('Rock-Paper-Scissors Self-Play', color='white', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Non-transitivity demonstration
# Create three fixed policies: R-heavy, P-heavy, S-heavy

def expected_reward(p: np.ndarray, q: np.ndarray) -> float:
    """Expected reward for row player with policy p against column player with policy q."""
    return float(p @ RPS_REWARD @ q)


policy_R = np.array([0.7, 0.15, 0.15])  # Rock-heavy
policy_P = np.array([0.15, 0.7, 0.15])  # Paper-heavy
policy_S = np.array([0.15, 0.15, 0.7])  # Scissors-heavy
policy_U = np.array([1/3, 1/3, 1/3])    # Uniform (Nash)

policies = {'Rock-heavy': policy_R, 'Paper-heavy': policy_P,
             'Scissors-heavy': policy_S, 'Uniform': policy_U}

names = list(policies.keys())
payoff_matrix = np.zeros((4, 4))
for i, (n1, p1) in enumerate(policies.items()):
    for j, (n2, p2) in enumerate(policies.items()):
        payoff_matrix[i, j] = expected_reward(p1, p2)

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor('#0f1117')
im = ax.imshow(payoff_matrix, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(names, rotation=30, ha='right', color='white')
ax.set_yticklabels(names, color='white')
ax.set_title('Expected Payoff Matrix (row vs col)', color='white')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{payoff_matrix[i,j]:.2f}", ha='center', va='center',
                fontsize=10, color='black', fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Non-transitivity:")
print(f"  Paper beats Rock: {expected_reward(policy_P, policy_R):.2f}")
print(f"  Scissors beats Paper: {expected_reward(policy_S, policy_P):.2f}")
print(f"  Rock beats Scissors: {expected_reward(policy_R, policy_S):.2f}")
print(f"  Uniform ties everyone: {expected_reward(policy_U, policy_R):.2f}, "
      f"{expected_reward(policy_U, policy_P):.2f}, {expected_reward(policy_U, policy_S):.2f}")

<div style="background: #1b2838; padding: 16px; border-radius: 8px; border-left: 4px solid #40c4ff;">
  <h2 style="color: #40c4ff; margin: 0;">Часть 4 — Кооперативная задача с общим вознаграждением</h2>
</div>

Двум агентам нужно выбрать одно и то же действие (скоординироваться), чтобы получить общее вознаграждение.
Это **чистая координационная игра** — оба агента делят сигнал вознаграждения.

In [ ]:
# Coordination game: 3 options (A, B, C)
# Reward = +1 if both agents choose the same action, 0 otherwise
# One action is 'focal' (labeled 0) giving reward +2 if both choose it

N_ACTIONS = 3
COORD_PAYOFF = np.array([2, 1, 1])  # reward if both choose action i


class CoopAgent:
    def __init__(self, alpha=0.1, epsilon=0.2):
        self.Q = np.zeros(N_ACTIONS)
        self.alpha = alpha
        self.epsilon = epsilon

    def act(self) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, N_ACTIONS - 1)
        return int(np.argmax(self.Q))

    def update(self, action: int, reward: float):
        self.Q[action] += self.alpha * (reward - self.Q[action])


def run_coordination(n_episodes: int = 5000, epsilon_decay: bool = True):
    a1, a2 = CoopAgent(), CoopAgent()
    coord_hist = []
    reward_hist = []

    for ep in range(n_episodes):
        if epsilon_decay:
            eps = max(0.01, 0.3 * (0.9995 ** ep))
            a1.epsilon = a2.epsilon = eps

        act1 = a1.act()
        act2 = a2.act()
        if act1 == act2:
            r = COORD_PAYOFF[act1]
        else:
            r = 0.0

        a1.update(act1, r)
        a2.update(act2, r)
        coord_hist.append(float(act1 == act2))
        reward_hist.append(r)

    return coord_hist, reward_hist, a1, a2


coord_hist, reward_hist, ca1, ca2 = run_coordination(8000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')

ax = axes[0]
ax.set_facecolor('#0f1117')
sm = smooth(coord_hist, 100)
ax.plot(sm, color='#40c4ff', lw=2)
ax.axhline(1/3, color='gray', linestyle=':', lw=1, label='Random coordination rate')
ax.set_xlabel('Episode'); ax.set_ylabel('Coordination Rate')
ax.set_title('Coordination Rate over Training', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')

ax = axes[1]
ax.set_facecolor('#0f1117')
action_names = ['Action A (focal)', 'Action B', 'Action C']
q1 = np.exp(ca1.Q - ca1.Q.max()); q1 /= q1.sum()
q2 = np.exp(ca2.Q - ca2.Q.max()); q2 /= q2.sum()
x = np.arange(3)
ax.bar(x - 0.2, q1, 0.35, label='Agent 1', color='#40c4ff', edgecolor='white')
ax.bar(x + 0.2, q2, 0.35, label='Agent 2', color='#ffb74d', edgecolor='white')
ax.set_xticks(x); ax.set_xticklabels(action_names, color='white')
ax.set_ylabel('Softmax Probability'); ax.set_title('Final Action Preferences', color='white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
for item in [ax.xaxis.label, ax.yaxis.label] + ax.get_yticklabels():
    item.set_color('white')

plt.suptitle('Cooperative Coordination Game with Shared Reward', color='white', fontsize=14)
plt.tight_layout()
plt.show()
print(f"Final coordination rate (last 1000 ep): {np.mean(coord_hist[-1000:]):.1%}")

## Сводка и ключевые выводы

| Эксперимент | Ключевой результат |
|------------|------------|
| IPD с Q-обучением | Агенты учатся кооперировать, получая больший совместный выигрыш, чем в (D,D) Нэше |
| TfT-оппонент | Q-агент быстро обнаруживает, что кооперация — оптимальный ответ на TfT |
| RPS self-play | Стратегия сходится примерно к равномерной (равновесие Нэша) |
| Нетранзитивность | Камень→Ножницы→Бумага→Камень: глобально лучшей стратегии в RPS нет |
| Координационная игра | Общее вознаграждение заставляет агентов координировать выбор |

**Ключевой вывод**: многоагентное RL порождает принципиально новые явления — возникновение кооперации, стратегическую взаимозависимость и нетранзитивные динамики, — которых нет в одноагентном RL.